## Neural Networks - Deep Learning
**Theodora Tzina - AEM: 10715**
### Exercise 3
#### ***Part C: Autoencoder - Digit Adder Reconstruction***

This notebook implements autoencoders that learn to add two MNIST digits.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import time
import tensorflow as tf
import matplotlib.pyplot as plt
import dataProccessing as dp
import digitAdder as da

from keras import layers, Model
from keras.callbacks import EarlyStopping
from keras.optimizers import Adam

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

**MNIST dataset preparation**

In [ ]:
# Load MNIST
x_train, y_train, x_test, y_test = dp.load_mnist()

# Create digit addition dataset
train_inputs, train_outputs, train_sums = da.create_digit_adder_dataset(
    x_train, y_train, seed=42
)

test_inputs, test_outputs, test_sums = da.create_digit_adder_dataset(
    x_test, y_test, seed=123
)

# Prepare data for different model types
# For convolutional models: (N, 28, 56, 1)
train_inputs_cnn = train_inputs.reshape(-1, 28, 56, 1)
train_outputs_cnn = train_outputs.reshape(-1, 28, 56, 1)
test_inputs_cnn = test_inputs.reshape(-1, 28, 56, 1)
test_outputs_cnn = test_outputs.reshape(-1, 28, 56, 1)

# For dense models: (N, 28*56)
train_inputs_flat = train_inputs.reshape(-1, 28*56)
train_outputs_flat = train_outputs.reshape(-1, 28*56)
test_inputs_flat = test_inputs.reshape(-1, 28*56)
test_outputs_flat = test_outputs.reshape(-1, 28*56)

print(f"\nDataset ready!")
print(f"Training samples: {len(train_inputs)}")
print(f"Test samples: {len(test_inputs)}")

## Visualization Helper Functions

In [ ]:
def plot_addition_examples(inputs, outputs, predictions, n=5):
    """Plot addition examples with predictions."""
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    
    for i in range(n):
        # Reshape if needed
        inp = inputs[i].reshape(28, 56)
        out = outputs[i].reshape(28, 56)
        pred = predictions[i].reshape(28, 56)
        
        # Input (A + B)
        axes[i, 0].imshow(inp, cmap='gray')
        axes[i, 0].set_title('Input: A + B')
        axes[i, 0].axis('off')
        
        # Target output
        axes[i, 1].imshow(out, cmap='gray')
        axes[i, 1].set_title('Target Sum')
        axes[i, 1].axis('off')
        
        # Predicted output
        axes[i, 2].imshow(pred, cmap='gray')
        axes[i, 2].set_title('Predicted Sum')
        axes[i, 2].axis('off')
    
    plt.tight_layout()
    plt.show()

## Model 1: Dense Autoencoder for Digit Addition

In [ ]:
# Model 1: Simple Dense Autoencoder
encoding_dim = 256

# Encoder
encoder_input = layers.Input(shape=(28*56,))
x = layers.Dense(1024, activation='relu')(encoder_input)
x = layers.Dense(512, activation='relu')(x)
latent = layers.Dense(encoding_dim, activation='relu')(x)

encoder1 = Model(encoder_input, latent, name='encoder1')

# Decoder
decoder_input = layers.Input(shape=(encoding_dim,))
x = layers.Dense(512, activation='relu')(decoder_input)
x = layers.Dense(1024, activation='relu')(x)
decoder_output = layers.Dense(28*56, activation='sigmoid')(x)

decoder1 = Model(decoder_input, decoder_output, name='decoder1')

# Full autoencoder
autoencoder1 = Model(encoder_input, decoder1(encoder1(encoder_input)), name='adder_ae1')
autoencoder1.compile(optimizer='adam', loss='mse')

print("\nModel 1: Dense Autoencoder")
print(f"Encoding dimension: {encoding_dim}")
print(f"Total parameters: {autoencoder1.count_params():,}")

# Train
print("\nTraining Model 1...")
start_time = time.time()
history1 = autoencoder1.fit(
    train_inputs_flat, train_outputs_flat,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    verbose=1
)
train_time1 = time.time() - start_time
print(f"Training time: {train_time1:.2f}s")

# Evaluate
test_loss1 = autoencoder1.evaluate(test_inputs_flat, test_outputs_flat, verbose=0)
print(f"Test MSE: {test_loss1:.6f}")

# Visualize
plot_training_history(history1, 'Model 1: Dense AE')
predictions1 = autoencoder1.predict(test_inputs_flat[:10])
plot_addition_examples(test_inputs_flat[:10], test_outputs_flat[:10], predictions1)

## Model 2: Deep Dense Autoencoder

In [ ]:
# Model 2: Deeper Dense Autoencoder
encoding_dim = 512

# Encoder
encoder_input = layers.Input(shape=(28*56,))
x = layers.Dense(2048, activation='relu')(encoder_input)
x = layers.BatchNormalization()(x)
x = layers.Dense(1024, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation='relu')(x)
latent = layers.Dense(encoding_dim, activation='relu')(x)

encoder2 = Model(encoder_input, latent, name='encoder2')

# Decoder
decoder_input = layers.Input(shape=(encoding_dim,))
x = layers.Dense(512, activation='relu')(decoder_input)
x = layers.BatchNormalization()(x)
x = layers.Dense(1024, activation='relu')(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(2048, activation='relu')(x)
decoder_output = layers.Dense(28*56, activation='sigmoid')(x)

decoder2 = Model(decoder_input, decoder_output, name='decoder2')

# Full autoencoder
autoencoder2 = Model(encoder_input, decoder2(encoder2(encoder_input)), name='adder_ae2')
autoencoder2.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

print("\nModel 2: Deep Dense Autoencoder")
print(f"Encoding dimension: {encoding_dim}")
print(f"Total parameters: {autoencoder2.count_params():,}")

# Train with early stopping
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

print("\nTraining Model 2...")
start_time = time.time()
history2 = autoencoder2.fit(
    train_inputs_flat, train_outputs_flat,
    epochs=50,
    batch_size=256,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)
train_time2 = time.time() - start_time
print(f"Training time: {train_time2:.2f}s")

# Evaluate
test_loss2 = autoencoder2.evaluate(test_inputs_flat, test_outputs_flat, verbose=0)
print(f"Test MSE: {test_loss2:.6f}")

# Visualize
plot_training_history(history2, 'Model 2: Deep Dense AE')
predictions2 = autoencoder2.predict(test_inputs_flat[:10])
plot_addition_examples(test_inputs_flat[:10], test_outputs_flat[:10], predictions2)

## Model 3: Convolutional Autoencoder for Digit Addition

In [ ]:
# Model 3: Convolutional Autoencoder
# Input: (28, 56, 1) - two digits side by side

# Encoder
encoder_input = layers.Input(shape=(28, 56, 1))
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(encoder_input)
x = layers.MaxPooling2D((2, 2), padding='same')(x)  # (14, 28, 32)
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2), padding='same')(x)  # (7, 14, 64)
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
latent = layers.MaxPooling2D((2, 2), padding='same')(x)  # (4, 7, 128)

encoder3 = Model(encoder_input, latent, name='encoder3')

# Decoder
decoder_input = layers.Input(shape=(4, 7, 128))
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(decoder_input)
x = layers.UpSampling2D((2, 2))(x)  # (8, 14, 128)
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.UpSampling2D((2, 2))(x)  # (16, 28, 64)
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = layers.UpSampling2D((2, 2))(x)  # (32, 56, 32)
# Crop to match output size (28, 56, 1)
x = layers.Cropping2D(cropping=((2, 2), (0, 0)))(x)  # (28, 56, 32)
decoder_output = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

decoder3 = Model(decoder_input, decoder_output, name='decoder3')

# Full autoencoder
autoencoder3 = Model(encoder_input, decoder3(encoder3(encoder_input)), name='adder_ae3')
autoencoder3.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

print("\nModel 3: Convolutional Autoencoder")
print(f"Total parameters: {autoencoder3.count_params():,}")

# Train
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

print("\nTraining Model 3...")
start_time = time.time()
history3 = autoencoder3.fit(
    train_inputs_cnn, train_outputs_cnn,
    epochs=50,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)
train_time3 = time.time() - start_time
print(f"Training time: {train_time3:.2f}s")

# Evaluate
test_loss3 = autoencoder3.evaluate(test_inputs_cnn, test_outputs_cnn, verbose=0)
print(f"Test MSE: {test_loss3:.6f}")

# Visualize
plot_training_history(history3, 'Model 3: Conv AE')
predictions3 = autoencoder3.predict(test_inputs_cnn[:10])
plot_addition_examples(test_inputs_cnn[:10], test_outputs_cnn[:10], predictions3)

## Model 4: Advanced Convolutional Autoencoder with Residual Connections

In [ ]:
# Model 4: Advanced Conv AE with skip connections
from keras.layers import Add, Concatenate

# Encoder with skip connections
encoder_input = layers.Input(shape=(28, 56, 1))

# Block 1
x1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(encoder_input)
x1 = layers.BatchNormalization()(x1)
x1 = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x1)
p1 = layers.MaxPooling2D((2, 2), padding='same')(x1)  # (14, 28, 32)

# Block 2
x2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
x2 = layers.BatchNormalization()(x2)
x2 = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x2)
p2 = layers.MaxPooling2D((2, 2), padding='same')(x2)  # (7, 14, 64)

# Block 3
x3 = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
x3 = layers.BatchNormalization()(x3)
latent = layers.MaxPooling2D((2, 2), padding='same')(x3)  # (4, 7, 128)

encoder4 = Model(encoder_input, [latent, x2, x1], name='encoder4')

# Decoder with skip connections
decoder_input = layers.Input(shape=(4, 7, 128))
skip2 = layers.Input(shape=(7, 14, 64))
skip1 = layers.Input(shape=(14, 28, 32))

# Up block 1
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(decoder_input)
x = layers.UpSampling2D((2, 2))(x)  # (8, 14, 128)
x = layers.Cropping2D(cropping=((1, 0), (0, 0)))(x)  # (7, 14, 128)
x = Concatenate()([x, skip2])  # Concat with skip connection

# Up block 2
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.UpSampling2D((2, 2))(x)  # (14, 28, 64)
x = Concatenate()([x, skip1])  # Concat with skip connection

# Up block 3
x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
x = layers.BatchNormalization()(x)
x = layers.UpSampling2D((2, 2))(x)  # (28, 56, 32)
decoder_output = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same')(x)

decoder4 = Model([decoder_input, skip2, skip1], decoder_output, name='decoder4')

# Full autoencoder
enc_out = encoder4(encoder_input)
dec_out = decoder4(enc_out)
autoencoder4 = Model(encoder_input, dec_out, name='adder_ae4')
autoencoder4.compile(optimizer=Adam(learning_rate=0.001), loss='mse')

print("\nModel 4: Advanced Conv AE with Skip Connections")
print(f"Total parameters: {autoencoder4.count_params():,}")

# Train
early_stop = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

print("\nTraining Model 4...")
start_time = time.time()
history4 = autoencoder4.fit(
    train_inputs_cnn, train_outputs_cnn,
    epochs=50,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)
train_time4 = time.time() - start_time
print(f"Training time: {train_time4:.2f}s")

# Evaluate
test_loss4 = autoencoder4.evaluate(test_inputs_cnn, test_outputs_cnn, verbose=0)
print(f"Test MSE: {test_loss4:.6f}")

# Visualize
plot_training_history(history4, 'Model 4: Advanced Conv AE')
predictions4 = autoencoder4.predict(test_inputs_cnn[:10])
plot_addition_examples(test_inputs_cnn[:10], test_outputs_cnn[:10], predictions4)

## Model Comparison

In [ ]:
# Compare all models
import pandas as pd

results = {
    'Model': [
        'Model 1: Dense AE',
        'Model 2: Deep Dense AE',
        'Model 3: Conv AE',
        'Model 4: Advanced Conv AE'
    ],
    'Parameters': [
        autoencoder1.count_params(),
        autoencoder2.count_params(),
        autoencoder3.count_params(),
        autoencoder4.count_params()
    ],
    'Test MSE': [
        test_loss1,
        test_loss2,
        test_loss3,
        test_loss4
    ],
    'Training Time (s)': [
        train_time1,
        train_time2,
        train_time3,
        train_time4
    ]
}

df = pd.DataFrame(results)
print("\n" + "="*70)
print("MODEL COMPARISON")
print("="*70)
print(df.to_string(index=False))
print("="*70)

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MSE comparison
axes[0].bar(range(4), df['Test MSE'], color=['steelblue', 'orange', 'green', 'red'])
axes[0].set_xticks(range(4))
axes[0].set_xticklabels(df['Model'], rotation=45, ha='right')
axes[0].set_ylabel('Test MSE')
axes[0].set_title('Model Performance Comparison')
axes[0].grid(axis='y', alpha=0.3)

# Training time comparison
axes[1].bar(range(4), df['Training Time (s)'], color=['steelblue', 'orange', 'green', 'red'])
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(df['Model'], rotation=45, ha='right')
axes[1].set_ylabel('Training Time (seconds)')
axes[1].set_title('Training Time Comparison')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## Detailed Analysis: Example Predictions

In [ ]:
# Show more detailed examples for the best model
print("\nDetailed Analysis - Model 3 (Convolutional AE)")
print("="*60)

# Test on specific digit combinations
n_examples = 15
test_pred = autoencoder3.predict(test_inputs_cnn[:n_examples])

plot_addition_examples(
    test_inputs_cnn[:n_examples], 
    test_outputs_cnn[:n_examples], 
    test_pred, 
    n=n_examples
)

# Calculate MSE for each example
for i in range(min(10, n_examples)):
    mse = np.mean((test_outputs_cnn[i] - test_pred[i])**2)
    print(f"Example {i+1} - Sum: {test_sums[i]:2d} - MSE: {mse:.6f}")

## Save Best Model

In [ ]:
# Save the best performing model
best_model_idx = df['Test MSE'].idxmin()
print(f"\nBest model: {df.iloc[best_model_idx]['Model']}")
print(f"Test MSE: {df.iloc[best_model_idx]['Test MSE']:.6f}")

# Save model (choose the best one)
# autoencoder3.save('digit_adder_model.h5')
# print("Model saved as 'digit_adder_model.h5'")